# TP4 : De l'UDF au panda UDF (selon spark)

PySpark UDF (a.k.a User Defined Function) est l'une des fonctionnalités les plus utiles de Spark SQL & DataFrame qui est utilisée pour étendre les capacités de construction de PySpark.

1. Introduction à l'UDF

2. Introduction à Pandas UDF

1. Démarrer une session spark

(nous voulons démarrer une session locale et lui donner un nom).

Nous utiliserons la classe SparkSession du sous-module pyspark.sql.

Appeler sc le spark.sparkContext.

- Indiquez ce qui est défini par SparkSession et sparkContext.

- le master est 'local' (spécifier le nombre de cœurs) et il faut donner un nom à l'application.

In [ ]:
from datetime import datetime, date
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

from pyspark.sql.functions import col 
from pyspark.sql import Row

import findspark
findspark.init()

In [ ]:
spark = SparkSession.builder.master("local[*]").appName('demo_RDD').getOrCreate()
sc = spark.sparkContext

In [ ]:
print("pyspark version:",pyspark.__version__)
sc

Un port (4040) qu'il faut absolument regarder.

Il permet de suivre la progression d'un processus spark.

http://localhost:4040

## 1. User Defined Function¶

Les fonctions définies par l'utilisateur (UDF) sont des routines programmables par l'utilisateur qui agissent sur une ligne. 

**Note**: les UDF sont les opérations les plus coûteuses, donc utilisez-les seulement, vous n'avez pas le choix et lorsque c'est le cas.

Les UDF de PySpark sont similaires aux UDF des bases de données traditionnelles. Dans PySpark, vous créez une fonction dans une syntaxe Python et vous l'enveloppez avec PySpark SQL udf() ou vous l'enregistrez comme udf et vous l'utilisez sur DataFrame et SQL respectivement.

Avant de créer un UDF, faites vos recherches pour vérifier si la fonction similaire que vous voulez est déjà disponible dans Spark SQL Functions. PySpark SQL fournit plusieurs fonctions communes prédéfinies et de nombreuses nouvelles fonctions sont ajoutées à chaque version.

Maintenant, convertissez cette fonction en UDF en passant la fonction à PySpark SQL udf(), cette fonction est disponible dans le paquetage org.apache.spark.sql.functions.udf. 

Assurez-vous d'importer ce package avant de l'utiliser.

La fonction PySpark SQL udf() renvoie un objet de la classe org.apache.spark.sql.expressions.UserDefinedFunction.

pyspark.sql.functions.udf

[udf function]( https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html?highlight=udf#pyspark.sql.functions.udf)

### 1.1. Les imports

 

In [ ]:
# import udf
from pyspark.sql.functions import udf
# import des Types
from pyspark.sql.types import IntegerType, FloatType

In order to use convertCase() function on PySpark SQL, you need to register the function with PySpark by using spark.udf.register().



### 1.2 Création d'un dataframe

- créer 2 dataframes avec 2 colonnes (étiquette, valeur) d'une dizaine de lignes

La différence entre les 2 dataframe est que pour le premier, les valeurs "value" sont de type long ou int, et pour le second, les valeurs "value" sont de type double.

Le label ne peut prendre que 2 valeurs et apparaitre sur au moins 4 lignes.

In [ ]:
# DataFrame with int for value column
l1 = ???
df1 = spark.createDataFrame(l1,['label', 'value'])
# Display schema and dataframe
df1.printSchema()
df1.show()

# DataFrame with double for value column
l2 = ???
# Display schela and dataframe
df2.printSchema()
df2.show()

### 1.3. Comment utiliser l'UDF dans les cadres de données ?

- Création de fonctions udf

Nous commencerons par créer une fonction python qui élève un nombre au carré.

Convertissez la fonction python en une fonction udf en utilisant la fonction *udf*.

*nom_de_la_fonction_udf = udf(lambda z : python_fonction(z), mydataType())*

mydataType() peut être IntegerType(), FloatType(),...

- définir une fonction python appelée square1 et renvoyer s*s

- définir une fonction udf (squareUDF_int) qui aura une sortie de type IntegerType()

- définir une fonction udf (squareUDF_float) qui aura une sortie de type FloatType()

In [ ]:
# python function
def squared1(s):
  return s * s

# Converting function to UDF 
squareUDF_int = udf(lambda z: squared1(z),IntegerType())

# Converting function to UDF 
squareUDF_float = udf(lambda z: squared1(z),FloatType())

- Appliquer squareUDF_int et squareUDF_float aux deux dataframes.

Nous commencerons par utiliser *select* dans le dataframe.

- en utilisant *select* et *col*.


In [ ]:
# using squareUDF_int and squareUDF_float on df1
df1.select(col("label"), col("value"), squareUDF_int(col("value")).alias("squared")).show()

df1.select(col("label"), col("value"), squareUDF_float(col("value")).alias("squared")).show()

# using squareUDF_int and squareUDF_float on df2
df2.select(col("label"), col("value"), squareUDF_int(col("value")).alias("squared")).show()

df2.select(col("label"), col("value"), squareUDF_float(col("value")).alias("squared")).show()

- Commenter et analyser

- Utiliser *withColumn* à la place de *select* and *col*

In [ ]:
# using squareUDF_int and squareUDF_float on df1
df1.withColumn("squared1",squareUDF_int(col("value"))).show()

df1.withColumn("squared1",squareUDF_float(col("value"))).show()

# using squareUDF_int and squareUDF_float on df2
df2.withColumn("squared1",squareUDF_int(col("value"))).show()

df2.withColumn("squared1",squareUDF_float(col("value"))).show()

- Commenter et analyser

### 1.4 "Registering" PySpark UDF et l'utiliser sur SQL

Afin d'utiliser la fonction convertCase() sur PySpark SQL, vous devez enregistrer la fonction avec PySpark en utilisant [spark.udf.register()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.UDFRegistration.register.html#pyspark.sql.UDFRegistration.register).

*spark.udf.register("name_function",pythonfunction, myType)*

pythonfuntion : fonction python précédemment définie ou remplacée par une opération lambda

myType : LonType() ou FloatType()

- enregistrer 2 fonctions "square2r_int" et "square2r_float"

Utiliser [*createOrReplaceTempView*](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.createOrReplaceTempView.html?highlight=pyspark.sql.dataframe.createorreplacetempview#pyspark.sql.DataFrame.createOrReplaceTempView) pour créer ou remplacer une vue temporaire locale avec les DataFrame df1 et df2.

- Ecrire et utiliser une [commande sql](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.sql.html#pyspark.sql.SparkSession.sql) en sélectionnant les colonnes label et valeur et en ajoutant une troisième colonne en élevant la colonne valeur au carré. Testez les deux fonctions registered sur les deux vues.


In [ ]:
from pyspark.sql.types import LongType

#python function
def squared2(s):
  return s * s

# registering
spark.udf.register("square2r_int", squared2, LongType())
spark.udf.register("square2r_float", squared2, FloatType())

# defining view
df1.createOrReplaceTempView("VAL_TABLE_1")
df2.createOrReplaceTempView("VAL_TABLE_2")

# application
spark.sql("select label, value, square2r_int(value) as Name from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, square2r_float(value) as Name from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, square2r_int(value) as Name from VAL_TABLE_2") \
     .show()

spark.sql("select label, value, square2r_float(value) as Name from VAL_TABLE_2") \
     .show()

### 1.5 Création d'un UDF à l'aide d'un decorator

Dans les sections précédentes, vous avez appris que la création d'un UDF est un processus en deux étapes, d'abord, vous devez créer une fonction Python, puis convertir la fonction en UDF en utilisant la fonction SQL [@udf(.)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html?highlight=pyspark.sql.functions.udf#pyspark.sql.functions.udf), cependant, vous pouvez éviter ces deux étapes et la créer en une seule étape en utilisant des annotations.

- Définissez deux fonctions (square) python , la première de type long et la seconde de type float et appliquez-les à l'aide de la fonction *withColumn* sur df1 et df2.

In [ ]:
@udf("long")
def square3(s):
  return s * s

@udf("float")
def square4(s):
  return s * s

df1.withColumn("squared3",square3(col("value"))).show()

df1.withColumn("squared3",square4(col("value"))).show()

df2.withColumn("squared3",square3(col("value"))).show()

df2.withColumn("squared3",square4(col("value"))).show()

### 1.6 Gestion de la vérification des valeurs nulles

Les UDF sont sources d'erreurs lorsqu'ils ne sont pas conçus avec soin. Par exemple, lorsque vous avez une colonne qui contient la valeur null dans certains enregistrements, il est préférable de vérifier la valeur null à l'intérieur d'une fonction UDF plutôt qu'en dehors.


Il est toujours préférable de vérifier la valeur null à l'intérieur d'une fonction UDF plutôt qu'à l'extérieur.
Dans tous les cas, si vous ne pouvez pas effectuer une vérification de nullité dans l'UDF, utilisez IF ou CASE WHEN pour vérifier la nullité et appeler l'UDF de manière conditionnelle.

Ici, remplacez les deux dernières valeurs par *None*

puis ajoutez une condition dans le registre (utilisez la fonction lambda et testez *z* avant d'utiliser *square*).

Testez la fonction de registre précédente. 

In [ ]:
# Les 2 dernières valeurs sont None
# DataFrame avec des value int
l1 = [(1, 0), (1, 8), (1, 3), (1, 3), (2, 1), (2, 9), (2, 6),\
      (2, 6), (2, None), (2, None)]
df1 = spark.createDataFrame(l1,['label', 'value'])
# Afficher le dataframe
df1.printSchema()
df1.show()

# DataFrame avec des value double
l2 = [(1, 0.), (1, 8.), (1, 3.), (1, 3.), (2, 1.), (2, 9.),\
      (2, 6.), (2, 6.), (2, None), (2, None)]
df2 = spark.createDataFrame(l2,['label', 'value'])
# Afficher le dataframe
df2.printSchema()
df2.show()

# gestion des null
spark.udf.register("_nullsafeSquareUDF_int", lambda z: squared2(z) if ?? else ?? , LongType())

df1.createOrReplaceTempView("VAL_TABLE_1")
df2.createOrReplaceTempView("VAL_TABLE_2")

# test avec gestion des null
spark.sql("select label, value, _nullsafeSquareUDF_int(value) as Name from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, _nullsafeSquareUDF_int(value) as Name from VAL_TABLE_2") \
     .show()

# test sans gestion des null
# La commande suivante déclanche une erreur
#spark.sql("select v2, value, square2r(value) as Name from VAL_TABLE_1") \
#     .show()

L'application sur une valeur d'un mauvais type entraine *null* mais l'application sur *null* entraine une erreur (sauf si on le gère).


# 2. Pandas UDF (pandas_udf) 

En utilisant la fonction pyspark.sql.functions.pandas_udf() vous pouvez créer une Pandas UDF (User Defined Function) qui est exécutée par PySpark avec Arrow pour transformer le DataFrame. PySpark fournit par défaut des centaines de fonctions intégrées, donc avant de créer votre propre fonction, je vous recommande de faire une petite recherche pour identifier si la fonction que vous créez est déjà disponible dans pyspark.sql.functions.

 Syntaxe
```
pandas_udf(f=None, returnType=None, functionType=None`
```

- f - Fonction définie par l'utilisateur
- returnType - Ceci est optionnel mais lorsque spécifié, il doit être soit une chaîne de type formatée DDL, soit n'importe quel type de pyspark.sql.types.DataType
- functionType - int, optionnel

### PySpark pandas_udf() Utilisation

La fonction pandas_udf() est une fonction intégrée de pyspark.sql.functions qui est utilisée pour créer la fonction Pandas définie par l'utilisateur et appliquer la fonction personnalisée à une colonne ou à l'ensemble du DataFrame.


### 2.1 Les *import*


In [1]:
# Imports 
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import *

import pandas as pd

### 2.2 Création d'un cadre de données

- créer/reprendre 2 dataframes précédentes


In [ ]:
# DataFrame avec des value int


# DataFrame avec des value double


### 2.3 Créer PySpark Pandas UDF

En utilisant [pandas_udf()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html?highlight=pandas_udf#pyspark.sql.functions.pandas_udf) nous allons créer la fonction UDF personnalisée. (utiliser *pandas_udf* au lieu de *udf*)

La fonction Python doit prendre une série pandas en entrée et retourner une série pandas de la même longueur, et vous devez les spécifier dans les indices de type Python. Spark exécute une UDF pandas en divisant les colonnes en lots, en appelant la fonction pour chaque lot en tant que sous-ensemble des données, puis en concaténant les résultats.
    

In [ ]:
# Declare the function and create the UDF
# 
# Create pandas_udf()
# Method 1 : using decorator
@pandas_udf(???)
#@pandas_udf('integer', PandasUDFType.SCALAR) # before Spark 3.0
def to_square_int(s: ???) -> ???:
    return s*s

@pandas_udf(FloatType())
#@pandas_udf('integer', PandasUDFType.SCALAR) # before Spark 3.0
def to_square_float(s: ???) -> ???:
    return s*s

# Method 2 : using pandas_udf function
def to_square1(s: ???) -> ???:
    return s*s
to_square_udf = pandas_udf(to_square1, returnType=????)


# Using UDF with select()
df1.withColumn("square",to_square_int(df1.value)).show()
df1.withColumn("square",to_square_float(df1.value)).show()

df2.withColumn("square",to_square_int(df2.value)).show()
df2.withColumn("square",to_square_float(df2.value)).show()

df1.select(df1.label,df1.value,to_square_udf(df1.value).alias("squared")).show()

In [ ]:
from scipy import stats

@pandas_udf('double')
def cdf(v):
    return pd.Series(stats.norm.cdf(v))

df1.withColumn('cumulative_probability', cdf(df1.value)).show()

nous soustrayons la moyenne de v de chaque valeur de v pour chaque groupe. 

La fonction de regroupement est définie par la fonction "groupby", c'est-à-dire que chaque DataFrame pandas.

DataFrame à la fonction définie par l'utilisateur a la même valeur "label". 

Les schémas d'entrée et de sortie de cette fonction définie par l'utilisateur sont les mêmes, nous passons donc "df.schema" au décorateur pandas_udf pour spécifier le schéma.

In [ ]:
print("df1")
df1.show()
print("df2")
df2.show()

print("df1 : mean1")

@pandas_udf(df1.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def mean1(pdf):
    return pdf.assign(value=pdf.value.mean())

# apply on df1
print(" mean1 on df1")
df1.groupby('label').apply(mean1).show()
# apply on df2
print(" mean1 on df2")
df2.groupby('label').apply(mean1).show()


print("df1 : substract mean1")

@pandas_udf(df1.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def subtract_mean1(pdf):
    return pdf.assign(value=pdf.value - pdf.value.mean())

df1.groupby('label').apply(subtract_mean1).show()
df2.groupby('label').apply(mean1).show()
print("df2")
df2.show()

print("df2 : mean")

@pandas_udf(df2.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def mean2(pdf):
    return pdf.assign(value=pdf.value.mean())
print(" df2 : mean2 on df1")
df1.groupby('label').apply(mean2).show()
print(" df2 : mean2 on df2")
df2.groupby('label').apply(mean2).show()

print("df2 : substract mean2")

@pandas_udf(df2.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def subtract_mean2(pdf):
    return pdf.assign(value=pdf.value - pdf.value.mean())

df2.groupby('label').apply(subtract_mean2).show()

- Expliquer le résultat précédent (df1 et df2 ont les mêmes valeurs ?)

# 3. test de temps

- créer un cadre de données avec un intervalle (10000 lignes)

La fonction square sort dans spark.

Comparer le temps d'exécution entre la fonction spark square , udf python square function, pandas udf python square function

In [ ]:
sc.stop()